```sql

Goal:
- simulate "online" scoring over a transaction stream
- compute velocity features incrementally (past-only)
- compare offline vs online feature values on the same rows (parity check)

Why:
If features differ between offline training and online serving, model quality will silently degrade

```

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
from sklearn.metrics import roc_auc_score, average_precision_score
from lightgbm import LGBMClassifier

#### Import data

In [3]:
train_df=pd.read_pickle('../data/interim/train_joined_split.pkl')

In [4]:
valid_df=pd.read_pickle('../data/interim/valid_joined_split.pkl')

In [5]:
train_df.shape, valid_df.shape

((472432, 434), (118108, 434))

In [6]:
#Sorting by Txn DT
train_df = train_df.sort_values("TransactionDT").reset_index(drop=True)
valid_df = valid_df.sort_values("TransactionDT").reset_index(drop=True)

In [7]:
base_features=["TransactionAmt",
               "ProductCD",
               "card1", "card2", "card3", "card4", "card5", "card6",
               "addr1", "addr2",
               "P_emaildomain", "R_emaildomain",
               "DeviceType", "DeviceInfo"
              ]
target_col="isFraud"

In [8]:
available_features = [c for c in base_features if c in train_df.columns]

In [9]:
# Adding some transformation on features
for df_ in [train_df,valid_df]:
    df_["log_txnamt"]=np.log1p(df_["TransactionAmt"]) #log1p works best for skewed and zero values well
    df_["has_identity"]=(df_[["DeviceType", "DeviceInfo"]].notna().any(axis=1).astype(int)) #Any one value is available

In [10]:
new_features = ["log_txnamt", "has_identity"]

In [11]:
feature_cols=available_features+new_features

In [12]:
# Simulating the online model predictions to find optimal threshold
X_train = train_df[feature_cols]
y_train = train_df[target_col]
X_valid = valid_df[feature_cols]
y_valid = valid_df[target_col]

# Basic preprocessing to handle nulls and types
cat_cols = X_train.select_dtypes(include=["object", "string", "category"]).columns.tolist()
num_cols = X_train.select_dtypes(include="number").columns.tolist()

for col in num_cols:
    X_train[col] = X_train[col].fillna(X_train[col].median())
    X_valid[col] = X_valid[col].fillna(X_valid[col].median())

for col in cat_cols:
    X_train[col] = X_train[col].fillna("Missing").astype(str).astype("category")
    X_valid[col] = X_valid[col].fillna("Missing").astype(str).astype("category")

# Train basic LightGBM model for simulation
model = LGBMClassifier(n_estimators=100, random_state=42, class_weight="balanced")
model.fit(X_train, y_train, categorical_feature=cat_cols)

# Score validation set
valid_pred = model.predict_proba(X_valid)[:, 1]

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Number of positive: 16599, number of negative: 455833
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.071589 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1808
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 16
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


In [13]:
import os

# Calculate Threshold Tradeoffs
thresholds = np.linspace(0.01, 0.99, 100)
precisions = [np.sum((valid_pred >= t) & (y_valid == 1)) / max(np.sum(valid_pred >= t), 1) for t in thresholds]
recalls = [np.sum((valid_pred >= t) & (y_valid == 1)) / max(np.sum(y_valid == 1), 1) for t in thresholds]
flagged_rates = [np.mean(valid_pred >= t) for t in thresholds]

threshold_metrics = pd.DataFrame({
    "threshold": thresholds,
    "precision": precisions,
    "recall": recalls,
    "flagged_rate": flagged_rates
})

os.makedirs("../results", exist_ok=True)
threshold_metrics.to_csv("../results/threshold_metrics.csv", index=False)
print("Threshold metrics exported successfully!")

Threshold metrics exported successfully!


#### End